# 08 — Paper Metrics Final Evaluation (GAP 1 / 2 / 3 / 5)

Supplementary evaluation on **test_final** (928 images / 6,398 pairs), executed after the 4 end-to-end runs have completed. This notebook is **read-only** with respect to previous experiment artifacts (samples CSV, model checkpoints, GT masks) and does not overwrite existing prediction files. All generated outputs are written to `outputs/metrics_final/<timestamp>/` along with execution logs.

| Section | Metric | Paper Usage |
|---|---|---|
| GAP 2 | Mask mAP50 / mAP50-95 for YOLO26-seg & YOLOv8n-seg on test_final | Main table, Mask mAP50 column |
| GAP 1 | Mask quality (IoU / Dice) across 4 pipelines vs. SAM reference masks | Mask quality table (RQ1) |
| GAP 3 | Paired Wilcoxon signed-rank test on absolute relative volume error | Statistical significance analysis |
| GAP 5 | Calorie estimation errors: MAE / RMSE / ME (kcal) | Secondary calorie metrics |

Evaluation Protocol (frozen across runs):
- Confidence thresholds: 04 = 0.05, 06a = 0.10, 06b = 0.05, 07 = 0.30 (fixed from test_tune sweeps).
- Ground truth masks for GAP 1: `data/processed/sam_masks_full/masks` (pseudo-GT reference).
- GAP 3 & 5 pair alignment: 4 CSV files share identical row order and sample indices (verified 6,398 / 6,398 matches).
- GPU execution (`device=0`) for Faster R-CNN, GrabCut, and SAM inference, matching notebooks 06a/06b.

**Execution workflow:** Run Cell 1 (setup) -> For each GAP, run the "SMOKE" cell first to verify outputs, then run the "FULL" cell. Detailed logs are written to `outputs/metrics_final/<timestamp>/run_<timestamp>.log`. Full execution takes ~30–40 minutes (predominantly GAP 1 GrabCut/SAM inference: ~1.8 s/image × 928).

**Side effects note:** Ultralytics `model.val()` may refresh `data/processed/yolo_ecustfd_seg/labels/val.cache`. This is an internal hash-based cache scan that does not affect data labels, predictions, or previous runs. No files in `outputs/predictions/` are modified.


In [1]:
import os, sys, time, json, csv, math, re, logging
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

os.chdir("E:/AI_Research/dlt8")
PROJECT = Path("E:/AI_Research/dlt8")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

RUN_TAG = time.strftime("%Y%m%d-%H%M%S")
OUT_DIR = PROJECT / "outputs/metrics_final" / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUT_DIR / f"run_{RUN_TAG}.log"

from src.e2e_pipeline.dataset_split import load_split, resolve_image_paths

IMAGESETS      = PROJECT / "data/raw/ECUSTFD/ImageSets/Main"
JPEG_DIR       = PROJECT / "data/raw/ECUSTFD/JPEGImages"
SAM_GT_DIR     = PROJECT / "data/processed/sam_masks_full/masks"
YOLO_DATA_ROOT = PROJECT / "data/processed/yolo_ecustfd_seg"

FROZEN_CONF = {"04": 0.05, "06a": 0.10, "06b": 0.05, "07": 0.30}
RUN_DIRS = {
    "04":  PROJECT / "outputs/predictions/04_e2e_paper_faithful_beta_20260903-090343",
    "06a": PROJECT / "outputs/predictions/06a_faster_rcnn_eval_20260903-131303",
    "06b": PROJECT / "outputs/predictions/06b_faster_rcnn_sam_eval_20260903-100830",
    "07":  PROJECT / "outputs/predictions/07_e2e_yolov8_paper_faithful_20260903-090734",
}
SAMPLES_CSV = {
    "04": "samples_test_final_conf5_beta.csv",
    "06a": "samples_test_final_conf10_beta.csv",
    "06b": "samples_test_final_conf5_beta.csv",
    "07": "samples_test_final_conf30_beta.csv",
}
CKPTS = {
    "04": PROJECT / "models/ecustfd_yolo26seg_best.pt",
    "07": PROJECT / "models/ecustfd_yolov8seg_best.pt",
}
FRCNN_CKPT = PROJECT / "models/faster_rcnn_new_best.pt"
YOLO_IMGSZ, YOLO_IOU, FRCNN_IOU = 480, 0.50, 0.50

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, encoding="utf-8"), logging.StreamHandler()],
)
for noisy in ("src.yolo_seg_eval.inference", "src.faster_rcnn.inference",
              "src.faster_rcnn.sam_inference", "yolo_seg_eval_pipeline"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
log = logging.getLogger("paper_metrics")
log.info("Metrics final run %s -> %s", RUN_TAG, OUT_DIR)

2026-09-06 15:21:55,221 INFO Metrics final run 20260906-152152 -> E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152


In [2]:
def test_final_images(limit: int | None = None) -> list[Path]:
    stems = load_split(IMAGESETS / "test_final.txt")
    paths = resolve_image_paths(stems, JPEG_DIR)
    if len(paths) != len(stems):
        raise RuntimeError(f"Resolved {len(paths)}/{len(stems)} test_final images")
    return paths[:limit] if limit else paths
def stem_of(p: Path) -> str:
    return p.stem


def class_of_stem(stem: str) -> str:
    m = re.match(r"^([a-zA-Z_]+?)\d", stem)
    return m.group(1) if m else stem


def norm_cls(c: str) -> str:
    return {"fired_dough_twist": "fried_dough_twist", "kiwi": "qiwi"}.get(c, c)


def iou_dice(pred: np.ndarray, gt: np.ndarray) -> tuple[float, float]:
    inter = int(np.logical_and(pred, gt).sum())
    p, g = int(pred.sum()), int(gt.sum())
    union = p + g - inter
    iou = inter / union if union > 0 else 0.0
    dice = 2 * inter / (p + g) if (p + g) > 0 else 0.0
    return iou, dice


def load_gt_food_mask(stem: str, shape: tuple[int, int]) -> np.ndarray | None:
    """Union of non-coin GT objects (SAM-mask GT), resized to `shape` if needed."""
    f = SAM_GT_DIR / f"{stem}.npy"
    if not f.exists():
        return None
    d = np.load(f, allow_pickle=True).item()
    gt = np.zeros(shape, dtype=bool)
    for o in d.get("objects", []):
        if o.get("class") == "coin":
            continue
        m = np.asarray(o["mask"]).astype(bool)
        if m.shape != shape:
            m = cv2_resize_nearest(m, shape)
        gt |= m
    return gt


def cv2_resize_nearest(mask: np.ndarray, shape: tuple[int, int]) -> np.ndarray:
    import cv2

    out = cv2.resize(mask.astype(np.uint8), (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return out.astype(bool)


def _load_rows(tag: str) -> list[dict]:
    """Read the frozen samples CSV of a model run (row i = pair i, verified order)."""
    return list(csv.DictReader(open(RUN_DIRS[tag] / SAMPLES_CSV[tag], encoding="utf-8")))


## GAP 2 — Mask mAP50 / mAP50-95 for YOLO26-seg (04) & YOLOv8n-seg (07) on test_final

Evaluated via Ultralytics `model.val()` on the 928 test_final images using original polygon annotations. Runtime is ~2–3 minutes on GPU.


In [3]:
# ---- GAP 2 + GAP 1 function definitions ----
def build_val_assets(out_dir: Path, limit: int | None = None) -> Path:
    """Write test_final val txt (absolute backslash paths) + dataset yaml."""
    stems = load_split(IMAGESETS / "test_final.txt")
    img_dir = YOLO_DATA_ROOT / "images/val"
    by_stem = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() == ".jpg"}
    missing = [s for s in stems if s not in by_stem]
    if missing:
        raise RuntimeError(f"{len(missing)} test_final stems missing in images/val, e.g. {missing[:3]}")
    stems = stems[:limit] if limit else stems

    # labels must exist via the standard images->labels substitution
    lab_dir = YOLO_DATA_ROOT / "labels/val"
    for s in stems:
        if not (lab_dir / f"{s}.txt").exists():
            raise RuntimeError(f"Missing polygon label for {s}")

    txt = out_dir / "test_final_val.txt"
    txt.write_text("\n".join(str(by_stem[s]) for s in stems), encoding="utf-8")

    yaml_path = out_dir / "ecustfd_seg_testfinal.yaml"
    yaml_path.write_text(
        "# Auto-generated: YOLO-seg val restricted to test_final (928 images)\n"
        f"path: {YOLO_DATA_ROOT.as_posix()}\n"
        f"train: {txt.as_posix()}\n"  # required key; val-only run never uses it
        f"val: {txt.as_posix()}\n"
        "names:\n"
        "  0: apple\n  1: banana\n  2: bread\n  3: bun\n  4: coin\n  5: doughnut\n"
        "  6: egg\n  7: fired_dough_twist\n  8: grape\n  9: lemon\n  10: litchi\n"
        "  11: mango\n  12: mooncake\n  13: orange\n  14: peach\n  15: pear\n"
        "  16: plum\n  17: qiwi\n  18: sachima\n  19: tomato\n",
        encoding="utf-8",
    )
    log.info("GAP2 assets: %s (%d images), %s", txt, len(stems), yaml_path)
    return yaml_path


def gap2_mask_map50(out_dir: Path, limit: int | None = None) -> dict:
    from src.yolo_seg_eval.inference import load_yolo_seg

    yaml_path = build_val_assets(out_dir, limit)
    results = {}
    for tag, ckpt in CKPTS.items():
        t0 = time.time()
        model = load_yolo_seg(ckpt, device=0)
        val_dir = out_dir / f"val_{tag}"
        m = model.val(
            data=str(yaml_path), split="val", imgsz=YOLO_IMGSZ, batch=4,
            device=0, plots=False, project=str(val_dir), name="val", exist_ok=True,
            verbose=False, workers=0,
        )
        res = {
            "mask_mAP50": float(m.seg.map50),
            "mask_mAP50_95": float(m.seg.map),
            "box_mAP50": float(m.box.map50),
        }
        try:
            names = m.names
            per_cls = {names[i]: float(x) for i, x in enumerate(m.seg.ap50) if x >= 0}
            res["mask_ap50_per_class"] = per_cls
        except Exception as e:  # noqa: BLE001
            log.warning("per-class mask AP50 unavailable for %s: %s", tag, e)
        results[tag] = res
        log.info("GAP2 %s (%s): mask_mAP50=%.4f box_mAP50=%.4f [%.1fs]",
                 tag, ckpt.name, res["mask_mAP50"], res["box_mAP50"], time.time() - t0)
        del model
    out = out_dir / "gap2_mask_map50.json"
    out.write_text(json.dumps(results, indent=2), encoding="utf-8")
    log.info("GAP2 saved -> %s", out)
    return results


# --------------------------------------------------------------------------- #
# GAP 1 — mask quality (IoU/Dice) of 4 pipelines vs SAM-mask GT
# --------------------------------------------------------------------------- #
def _yolo_backproject(mask_canvas: np.ndarray, orig_w: int, orig_h: int,
                      imgsz: int, stride: int = 32) -> np.ndarray:
    """Map a letterboxed YOLO mask back to original image size (nearest)."""
    r = min(imgsz / orig_h, imgsz / orig_w)
    new_w, new_h = round(orig_w * r), round(orig_h * r)
    dw, dh = imgsz - new_w, imgsz - new_h
    dw, dh = float(np.mod(dw, stride)), float(np.mod(dh, stride))
    top, left = dh / 2.0, dw / 2.0
    mh, mw = mask_canvas.shape
    ys = np.clip(np.round(np.arange(orig_h) * r + top).astype(int), 0, mh - 1)
    xs = np.clip(np.round(np.arange(orig_w) * r + left).astype(int), 0, mw - 1)
    return mask_canvas[np.ix_(ys, xs)]


def _paste_crop(full: np.ndarray, crop: np.ndarray, bbox) -> None:
    H, W = full.shape
    x1, y1, x2, y2 = bbox
    ix1, iy1 = max(0, int(round(x1))), max(0, int(round(y1)))
    ix2, iy2 = min(W, int(round(x2))), min(H, int(round(y2)))
    if ix2 <= ix1 or iy2 <= iy1:
        return
    ch, cw = iy2 - iy1, ix2 - ix1
    full[iy1:iy2, ix1:ix2] |= crop[:ch, :cw]


def _pred_union_from_dets(dets: list[dict], orig_shape, kind: str,
                          imgsz: int = YOLO_IMGSZ) -> np.ndarray | None:
    """Union of non-coin detection masks, in original image space."""
    H, W = orig_shape
    full = np.zeros((H, W), dtype=bool)
    n_food = 0
    for d in dets:
        if d.get("class_name") == "coin" or d.get("mask") is None:
            continue
        n_food += 1
        m = np.asarray(d["mask"]).astype(bool)
        if kind == "yolo":
            full |= _yolo_backproject(m, W, H, imgsz)
        else:  # grabcut / sam: bbox-cropped masks in original coords
            _paste_crop(full, m, d["bbox"])
    return full if n_food else None


def _load_models():
    from src.faster_rcnn.inference import load_faster_rcnn
    from src.faster_rcnn.sam_inference import load_sam
    from src.yolo_seg_eval.inference import load_yolo_seg

    models = {}
    models["04"] = ("yolo", load_yolo_seg(CKPTS["04"], device=0))
    models["07"] = ("yolo", load_yolo_seg(CKPTS["07"], device=0))
    models["06a"] = ("grabcut", load_faster_rcnn(FRCNN_CKPT, device=0))
    models["06b"] = ("sam", (load_faster_rcnn(FRCNN_CKPT, device=0), load_sam(device=0)))
    return models


In [4]:
# ---- Cell 3 [SMOKE 16 ảnh]: GAP 2 ----------------------------------------------
gap2 = gap2_mask_map50(OUT_DIR, limit=16)


2026-09-06 15:21:55,509 INFO GAP2 assets: E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\test_final_val.txt (16 images), E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\ecustfd_seg_testfinal.yaml


Ultralytics 8.4.87  Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)


YOLO26n-seg summary (fused): 139 layers, 2,692,784 parameters, 0 gradients, 9.0 GFLOPs


WARNING val: Slow image access detected (ping: 0.00.0 ms, read: 8.31.6 MB/s, size: 46.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/


val: Scanning E:\AI_Research\dlt8\data\processed\yolo_ecustfd_seg\labels\val... 16 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 16/16 797.2it/s 0.0s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 22.1it/s 0.2s

                   all         16         32          1          1      0.995      0.995          1          1      0.995      0.957


Speed: 0.2ms preprocess, 3.3ms inference, 0.0ms loss, 1.7ms postprocess per image


2026-09-06 15:22:01,517 INFO GAP2 07 (ecustfd_yolov8seg_best.pt): mask_mAP50=0.9950 box_mAP50=0.9950 [0.4s]


2026-09-06 15:22:01,520 INFO GAP2 saved -> E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\gap2_mask_map50.json


In [5]:
# ---- Cell 4 [FULL 928 ảnh]: GAP 2 ----------------------------------------------
gap2 = gap2_mask_map50(OUT_DIR, limit=None)


2026-09-06 15:22:01,542 INFO GAP2 assets: E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\test_final_val.txt (928 images), E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\ecustfd_seg_testfinal.yaml


Ultralytics 8.4.87  Python-3.14.4 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)


YOLO26n-seg summary (fused): 139 layers, 2,692,784 parameters, 0 gradients, 9.0 GFLOPs


val: Fast image access  (ping: 0.00.0 ms, read: 82.7179.9 MB/s, size: 40.1 KB)


val: Scanning E:\AI_Research\dlt8\data\processed\yolo_ecustfd_seg\labels\val... 99 images, 0 backgrounds, 0 corrupt: 10% ━─────────── 99/928 294.2it/s 0.1s<2.8s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 232/232 24.1it/s 9.6s

                   all        928       1888      0.936      0.938      0.971      0.955      0.936      0.938      0.971       0.94


Speed: 0.3ms preprocess, 3.1ms inference, 0.0ms loss, 1.3ms postprocess per image


2026-09-06 15:22:21,797 INFO GAP2 07 (ecustfd_yolov8seg_best.pt): mask_mAP50=0.9715 box_mAP50=0.9715 [9.8s]


2026-09-06 15:22:21,799 INFO GAP2 saved -> E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\gap2_mask_map50.json


## GAP 1 — Segmentation Mask Quality (IoU / Dice) across 4 Pipelines vs. SAM Reference

For each test_final image, all 4 pipelines (YOLO26-seg, YOLOv8n-seg, FR-CNN + GrabCut, and FR-CNN + SAM) are evaluated at their frozen confidence thresholds. The union of food instance masks (excluding coin) is evaluated against the reference food mask union. Results are exported to per-image CSV and summary JSON files (micro/macro per class).

**Interpretation note:** Because reference masks were generated using SAM with ground-truth bounding-box prompts, pipeline 06b (FR-CNN + SAM) has an inherent architectural alignment with the reference. In the paper, treat 06b as an upper-bound reference rather than an independent benchmark. Runtime is ~1.8 s/image (~28–35 minutes total, mostly GrabCut processing).


In [6]:
def gap1_mask_quality(out_dir: Path, limit: int | None = None) -> dict:
    import cv2

    from src.faster_rcnn.sam_inference import predict_one_with_sam
    from src.yolo_seg_eval.inference import predict_one as yolo_predict_one

    images = test_final_images(limit)
    models = _load_models()
    per_image: list[dict] = []
    t0 = time.time()

    for idx, img_path in enumerate(images):
        stem = stem_of(img_path)
        img = cv2.imread(str(img_path))
        if img is None:
            raise IOError(f"cv2.imread returned None: {img_path}")
        H, W = img.shape[:2]
        gt = load_gt_food_mask(stem, (H, W))
        cls = norm_cls(class_of_stem(stem))

        row = {"stem": stem, "class": cls, "gt_area": int(gt.sum()) if gt is not None else -1}
        gt_ok = gt is not None and gt.sum() > 0

        # YOLO pipelines
        for tag in ("04", "07"):
            kind = models[tag][0]
            dets = yolo_predict_one(models[tag][1], img_path, conf=FROZEN_CONF[tag],
                                    iou_threshold=YOLO_IOU, imgsz=YOLO_IMGSZ, device=0)
            pred = _pred_union_from_dets(dets, (H, W), "yolo")
            if not gt_ok:
                row[tag] = ""
            elif pred is None:
                row[tag] = "NO_PRED"
            else:
                i, dsc = iou_dice(pred, gt)
                row[tag] = f"{i:.4f}|{dsc:.4f}|{int(pred.sum())}"

        # FR-CNN + GrabCut
        dets = _frcnn_grabcut_one(models["06a"][1], img_path)
        pred = _pred_union_from_dets(dets, (H, W), "crop")
        row["06a"] = "" if not gt_ok else ("NO_PRED" if pred is None else
                                           f"{iou_dice(pred, gt)[0]:.4f}|{iou_dice(pred, gt)[1]:.4f}|{int(pred.sum())}")

        # FR-CNN + SAM
        frcnn, sam = models["06b"][1]
        dets = predict_one_with_sam(frcnn, sam, img_path, conf=FROZEN_CONF["06b"],
                                    iou_threshold=FRCNN_IOU, device=0)
        pred = _pred_union_from_dets(dets, (H, W), "crop")
        row["06b"] = "" if not gt_ok else ("NO_PRED" if pred is None else
                                           f"{iou_dice(pred, gt)[0]:.4f}|{iou_dice(pred, gt)[1]:.4f}|{int(pred.sum())}")

        per_image.append(row)
        if (idx + 1) % 25 == 0 or idx + 1 == len(images):
            log.info("GAP1 progress %d/%d [%.1fs]", idx + 1, len(images), time.time() - t0)

    csv_path = out_dir / "gap1_mask_quality_per_image.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["stem", "class", "gt_area", "04", "07", "06a", "06b"])
        w.writeheader()
        w.writerows(per_image)

    summary = _summarize_gap1(per_image)
    out = out_dir / "gap1_mask_quality_summary.json"
    out.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    log.info("GAP1 saved -> %s, %s", csv_path, out)
    return summary


def _frcnn_grabcut_one(model, img_path):
    from src.faster_rcnn.inference import predict_one as frcnn_predict_one

    return frcnn_predict_one(model, img_path, conf=FROZEN_CONF["06a"],
                             iou_threshold=FRCNN_IOU, device=0)


def _summarize_gap1(rows: list[dict]) -> dict:
    tags = ["04", "07", "06a", "06b"]
    per_cls: dict[str, dict[str, list[float]]] = defaultdict(lambda: defaultdict(list))
    counts = {t: {"n_eval": 0, "n_no_pred": 0, "n_empty_pred": 0} for t in tags}
    for r in rows:
        for t in tags:
            v = r[t]
            if v == "":
                continue
            if v == "NO_PRED":
                counts[t]["n_no_pred"] += 1
                continue
            iou, dice, area = (float(x) for x in v.split("|"))
            if area == 0:
                counts[t]["n_empty_pred"] += 1
            counts[t]["n_eval"] += 1
            per_cls[r["class"]][t].extend([iou, dice])
    summary = {}
    for t in tags:
        cls_rows = {}
        for c in sorted(per_cls):
            vals = per_cls[c][t]
            ious = vals[0::2]
            dices = vals[1::2]
            cls_rows[c] = {"n": len(ious),
                           "mean_iou": sum(ious) / len(ious),
                           "mean_dice": sum(dices) / len(dices)}
        all_ious = [v for c in per_cls for v in per_cls[c][t][0::2]]
        summary[t] = {
            **counts[t],
            "mean_iou_micro": sum(all_ious) / len(all_ious) if all_ious else None,
            "mean_iou_macro": (sum(v["mean_iou"] for v in cls_rows.values()) / len(cls_rows)
                               if cls_rows else None),
            "mean_dice_micro": (sum(v for c in per_cls for v in per_cls[c][t][1::2]) / len(all_ious)
                                if all_ious else None),
            "per_class": cls_rows,
        }
    return summary


In [7]:
# ---- Cell 5 [SMOKE 12 ảnh]: GAP 1 ----------------------------------------------
gap1 = gap1_mask_quality(OUT_DIR, limit=12)


2026-09-06 15:22:44,085 INFO GAP1 progress 12/12 [20.3s]


2026-09-06 15:22:44,087 INFO GAP1 saved -> E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\gap1_mask_quality_per_image.csv, E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\gap1_mask_quality_summary.json


In [8]:
# ---- Cell 6 [FULL 928 ảnh]: GAP 1 ----------------------------------------------
gap1 = gap1_mask_quality(OUT_DIR, limit=None)


2026-09-06 15:23:24,751 INFO GAP1 progress 25/928 [39.3s]


2026-09-06 15:24:15,670 INFO GAP1 progress 50/928 [90.2s]


2026-09-06 15:24:56,079 INFO GAP1 progress 75/928 [130.6s]


2026-09-06 15:25:28,752 INFO GAP1 progress 100/928 [163.3s]


2026-09-06 15:26:25,157 INFO GAP1 progress 125/928 [219.7s]


...

2026-09-06 15:47:27,348 INFO GAP1 progress 875/928 [1481.9s]


2026-09-06 15:48:09,573 INFO GAP1 progress 900/928 [1524.1s]


2026-09-06 15:48:53,359 INFO GAP1 progress 925/928 [1567.9s]


2026-09-06 15:48:58,608 INFO GAP1 progress 928/928 [1573.1s]


2026-09-06 15:48:58,617 INFO GAP1 saved -> E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\gap1_mask_quality_per_image.csv, E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\gap1_mask_quality_summary.json


## GAP 3 — Paired Wilcoxon Signed-Rank Test on Relative Volume Error

Evaluated on common pairs with valid ground truth across **all 4 models** (N = 6,334 / 6,398 pairs, identical row ordering). Compares 04 against 06a, 06b, and 07. Primary statistical test is reported at item-level (N = 41 items) to address multi-view clustering and avoid pseudo-replication.


In [9]:
def gap3_wilcoxon(out_dir: Path) -> dict:
    """Wilcoxon signed-rank, ITEM-LEVEL (primary) + pair-level (reference only).

    Pairs are NOT independent observations: all 6,398 pairs come from 45 physical
    food items (median 64 pairs/item, max 729). A pair-level test would massively
    inflate significance (pseudo-replication). The item-level test aggregates the
    |relative volume error| per item first (N = number of items with GT valid in
    all 4 models, ~41), which respects independence.
    Pair-level p-values are kept in the output ONLY as a documented reference; the
    paper must cite the item-level numbers.
    """
    from scipy.stats import wilcoxon

    data = {tag: _load_rows(tag) for tag in RUN_DIRS}
    n = {len(v) for v in data.values()}
    if len(n) != 1:
        raise RuntimeError(f"Row-count mismatch across CSVs: {n}")
    n_rows = next(iter(n))

    def valid(r):
        try:
            v = float(r["gt_volume_cm3"])
        except (TypeError, ValueError):
            return False
        return math.isfinite(v) and v > 0

    masks = [valid_rows(data[t]) for t in RUN_DIRS]
    keep = [all(m[i] for m in masks) for i in range(n_rows)]

    def pair_errs(tag):
        rows, out = data[tag], []
        for r, k in zip(rows, keep):
            if not k:
                continue
            v_r = float(r["gt_volume_cm3"])
            out.append(abs(float(r["v_tilde_cm3"]) - v_r) / v_r)
        return np.asarray(out)

    e = {t: pair_errs(t) for t in RUN_DIRS}
    # item-level aggregation: mean |err| per item (items kept only if GT-valid in ALL models)
    def item_errs(tag):
        agg = {}
        for r, k in zip(data[tag], keep):
            if not k:
                continue
            it = r["item_id"]
            v_r = float(r["gt_volume_cm3"])
            agg.setdefault(it, []).append(abs(float(r["v_tilde_cm3"]) - v_r) / v_r)
        return {it: float(np.mean(v)) for it, v in agg.items()}

    A = {t: item_errs(t) for t in RUN_DIRS}
    items = sorted(set.intersection(*[set(A[t]) for t in RUN_DIRS]))
    log.info("GAP3: %d pairs GT-valid in all 4 models; %d items after intersection", len(e["04"]), len(items))

    results = {
        "n_items": len(items),
        "n_pairs": int(len(e["04"])),
        "mean_abs_rel_err": {t: float(v.mean()) for t, v in e.items()},
        "note": "item-level is the reportable test (pairs are clustered within items); "
                "pair_level is reference only (pseudo-replicated)",
    }
    for a, b in [("04", "06a"), ("04", "06b"), ("04", "07")]:
        # item-level (primary)
        ea = np.array([A[a][i] for i in items])
        eb = np.array([A[b][i] for i in items])
        diff = ea - eb
        try:
            stat, p_item = wilcoxon(ea, eb, zero_method="wilcox", alternative="two-sided")
        except ValueError:
            stat, p_item = float("nan"), float("nan")
        # pair-level (reference only)
        pdiff = e[a] - e[b]
        _, p_pair = wilcoxon(e[a], e[b], zero_method="wilcox", alternative="two-sided")
        res = {
            "item_level": {
                "n": len(items),
                "mean_err_a": float(ea.mean()), "mean_err_b": float(eb.mean()),
                "p_value": float(p_item), "W": float(stat),
                "a_better_items": int((diff < 0).sum()), "b_better_items": int((diff > 0).sum()),
                "tie_items": int((diff == 0).sum()),
            },
            "pair_level_reference_only": {
                "n": len(e[a]),
                "p_value": float(p_pair),
                "a_better": int((pdiff < 0).sum()), "b_better": int((pdiff > 0).sum()),
            },
        }
        results[f"{a}_vs_{b}"] = res
        log.info("GAP3 %s vs %s: ITEM p=%.4f (N=%d) | [pair-level ref p=%.2e]",
                 a, b, p_item, len(items), p_pair)

    out = out_dir / "gap3_wilcoxon.json"
    out.write_text(json.dumps(results, indent=2), encoding="utf-8")
    log.info("GAP3 saved -> %s", out)
    return results


def valid_rows(rows):
    out = []
    for r in rows:
        try:
            v = float(r["gt_volume_cm3"])
        except (TypeError, ValueError):
            v = float("nan")
        out.append(math.isfinite(v) and v > 0)
    return out

In [10]:
# ---- Cell 7 [FULL, không cần smoke — chạy ~2 s]: GAP 3 --------------------------
gap3 = gap3_wilcoxon(OUT_DIR)


2026-09-06 15:49:16,582 INFO GAP3: 6334 pairs GT-valid in all 4 models; 41 items after intersection


2026-09-06 15:49:16,591 INFO GAP3 04 vs 06a: ITEM p=0.4962 (N=41) | [pair-level ref p=6.86e-101]


2026-09-06 15:49:16,595 INFO GAP3 04 vs 06b: ITEM p=1.0000 (N=41) | [pair-level ref p=2.32e-88]


2026-09-06 15:49:16,597 INFO GAP3 04 vs 07: ITEM p=0.4486 (N=41) | [pair-level ref p=1.43e-02]


2026-09-06 15:49:16,599 INFO GAP3 saved -> E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\gap3_wilcoxon.json


## GAP 5 — Calorie Estimation Errors: MAE / RMSE (kcal)

Convention: `kcal_true = (q_gt / density_gt) * gt_mass_g` and `kcal_pred = (q_pred / density_pred) * mass_g`.
The reference side uses GT-class parameters because `gt_mass_g` is the measured truth.
The prediction side uses predicted-class parameters end-to-end.
This matches the original ECUSTFD MATLAB detected-class loop: `calorie(i) = volume * food_info{i+1,3}` in `faster_rcnn_rec.m` (~line 7071, loop over the detector's `boxes_cell`).
Thus, a misclassified accepted prediction carries the calorie density and mass density of its predicted class.
Undetected pairs (miss penalties) still use `kcal_pred = 0`.
The CSV `kcal` column is not used because it was computed before beta volume calibration.
This replaces the earlier GT-class hybrid (2026-09-06 fix), which combined predicted-class mass with GT-class calorie parameters.

In [11]:
def gap5_kcal(out_dir: Path) -> dict:
    """Calorie MAE/RMSE/ME with a single paper-faithful definition.

    Ground truth uses GT-class q_kcal_per_cm3 and density with the measured gt_mass_g:
        kcal_true = (q_gt / density_gt) * gt_mass_g
    Predictions use predicted-class parameters end-to-end, matching the detected-class
    loop in faster_rcnn_rec.m (line 156):
        kcal_pred = (q_pred / density_pred) * mass_g
    where mass_g = beta-calibrated volume * predicted-class density. Penalty rows
    (complete miss) contribute kcal_pred = 0 (penalized convention). n_eval excludes
    rows with invalid GT targets or missing required class parameters; penalty rows
    with valid GT targets remain in N. The CSV `kcal` column itself is NOT used: it
    was computed from the pre-beta volume (estimate_calorie_runtime runs before
    calibrate_volume in the e2e pipelines), which would mix two inconsistent
    definitions. GT-class parameters are an oracle only on the reference side,
    required because gt_mass is the measured truth.
    """
    from src.calorie_estimation.food_info_xls import parse_food_info
    from src.constants import DENSITY_G_CM3

    fi = parse_food_info(PROJECT / "data/raw/ECUSTFD/paper/food_info.xls")
    alias = {"fired_dough_twist": "fried_dough_twist", "kiwi": "qiwi"}

    data = {tag: _load_rows(tag) for tag in RUN_DIRS}
    results = {"definition": "kcal_true = (q_gt / density_gt) * gt_mass; "
                             "kcal_pred = (q_pred / density_pred) * mass_g; "
                             "q from food_info.xls (paper Table 1), with predicted-class "
                             "parameters for accepted predictions",
               "per_model": {}}
    for tag, rows in data.items():
        errs, n_miscls, n_skipped = [], 0, 0
        for r in rows:
            try:
                gt_mass = float(r["gt_mass_g"])
                gt_vol = float(r["gt_volume_cm3"])
            except (TypeError, ValueError):
                continue
            if not (math.isfinite(gt_mass) and gt_mass > 0 and math.isfinite(gt_vol) and gt_vol > 0):
                continue
            gt_cls = r["gt_class"] or r["class_name"]
            gt_cls = alias.get(gt_cls, gt_cls)
            pred_cls = alias.get(r["class_name"], r["class_name"])
            try:
                q_gt = float(fi[gt_cls]["kcal_per_cm3"])
                dens_gt = float(DENSITY_G_CM3.get(gt_cls, gt_mass / gt_vol))
                kcal_true = (q_gt / dens_gt) * gt_mass
                if r["top_path"] == "":
                    kcal_pred = 0.0
                else:
                    q_pred = float(fi[pred_cls]["kcal_per_cm3"])
                    dens_pred = float(DENSITY_G_CM3[pred_cls])
                    kcal_pred = (q_pred / dens_pred) * float(r["mass_g"])
            except KeyError:
                n_skipped += 1
                continue
            errs.append(kcal_pred - kcal_true)
            if r["top_path"] != "" and pred_cls != gt_cls:
                n_miscls += 1
        e = np.asarray(errs)
        res = {
            "n_eval": int(len(e)),
            "mae_kcal": float(np.abs(e).mean()),
            "rmse_kcal": float(np.sqrt((e ** 2).mean())),
            "me_kcal": float(e.mean()),
            "n_misclassified_accepted": int(n_miscls),
            "n_skipped": int(n_skipped),
        }
        results["per_model"][tag] = res
        log.info("GAP5 %s: N=%d MAE=%.2f kcal RMSE=%.2f kcal ME=%+.2f kcal",
                 tag, res["n_eval"], res["mae_kcal"], res["rmse_kcal"], res["me_kcal"])
        if n_skipped:
            log.warning("GAP5 %s: skipped %d rows with missing class parameters", tag, n_skipped)

    out = out_dir / "gap5_kcal.json"
    out.write_text(json.dumps(results, indent=2), encoding="utf-8")
    log.info("GAP5 saved -> %s", out)
    return results

In [12]:
# ---- Cell 8 [FULL, chạy ~7 s]: GAP 5 --------------------------------------------
gap5 = gap5_kcal(OUT_DIR)


2026-09-06 15:49:16,727 INFO GAP5 04: N=6334 MAE=41.60 kcal RMSE=107.97 kcal ME=-13.32 kcal


2026-09-06 15:49:16,735 INFO GAP5 06a: N=6334 MAE=65.01 kcal RMSE=249.85 kcal ME=-38.55 kcal


2026-09-06 15:49:16,743 INFO GAP5 06b: N=6334 MAE=63.57 kcal RMSE=249.81 kcal ME=-39.42 kcal


2026-09-06 15:49:16,752 INFO GAP5 07: N=6334 MAE=40.29 kcal RMSE=99.71 kcal ME=-13.37 kcal


2026-09-06 15:49:16,754 INFO GAP5 saved -> E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\gap5_kcal.json


## Summary Table — Run after all FULL cells above complete

Formats the consolidated metrics into paper main-table format for inclusion in results sections.


In [13]:
# ---- Cell 9 [SUMMARY]: ghép kết quả các GAP ------------------------------------
summary = {}
# (key hiển thị, tên file JSON thực tế do các GAP ghi)
FILES = {
    "gap2_mask_map50":   "gap2_mask_map50.json",
    "gap1_mask_quality": "gap1_mask_quality_summary.json",
    "gap3_wilcoxon":     "gap3_wilcoxon.json",
    "gap5_kcal":         "gap5_kcal.json",
}
for key, fname in FILES.items():
    f = OUT_DIR / fname
    if f.exists():
        summary[key] = json.loads(f.read_text(encoding="utf-8"))
    else:
        log.warning("Thiếu %s (cell FULL của GAP tương ứng chưa chạy?)", f)
        summary[key] = None

(OUT_DIR / "summary_all_gaps.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

hdr = f"{'Model':22s} {'Mask mAP50':>10s} {'IoU micro':>10s} {'Dice micro':>11s} {'MAE kcal':>9s} {'RMSE kcal':>10s}"
print(hdr)
print("-" * len(hdr))
for tag, name in [("04", "YOLO26n-Seg (Ours)"), ("07", "YOLOv8n-Seg"),
                  ("06a", "FR-CNN+GrabCut"), ("06b", "FR-CNN+SAM")]:
    g2 = ((summary.get("gap2_mask_map50") or {}).get(tag) or {})
    g1 = ((summary.get("gap1_mask_quality") or {}).get(tag) or {})
    g5 = ((summary.get("gap5_kcal") or {}).get("per_model") or {}).get(tag) or {}
    map50 = f"{g2['mask_mAP50']*100:.2f}" if g2.get("mask_mAP50") is not None else "  —"
    iou   = f"{g1['mean_iou_micro']*100:.2f}" if g1.get("mean_iou_micro") is not None else "  —"
    dice  = f"{g1['mean_dice_micro']*100:.2f}" if g1.get("mean_dice_micro") is not None else "  —"
    mae   = f"{g5['mae_kcal']:.2f}" if g5.get("mae_kcal") is not None else "  —"
    rmse  = f"{g5['rmse_kcal']:.2f}" if g5.get("rmse_kcal") is not None else "  —"
    print(f"{name:22s} {map50:>10s} {iou:>10s} {dice:>11s} {mae:>9s} {rmse:>10s}")

print()
g3 = summary.get("gap3_wilcoxon") or {}
if g3:
    print(f"GAP3 Wilcoxon ITEM-LEVEL (N={g3.get('n_items')} items, primary — pairs are clustered):")
    for cmp_name in ["04_vs_06a", "04_vs_06b", "04_vs_07"]:
        r = g3.get(cmp_name)
        if r:
            it = r["item_level"]
            print(f"  {cmp_name}: item p={it['p_value']:.4f} | 04 better {it['a_better_items']}, "
                  f"baseline better {it['b_better_items']} "
                  f"(pair-level ref p={r['pair_level_reference_only']['p_value']:.2e})")
log.info("Summary written -> %s", OUT_DIR / "summary_all_gaps.json")

2026-09-06 15:49:16,778 INFO Summary written -> E:\AI_Research\dlt8\outputs\metrics_final\20260906-152152\summary_all_gaps.json


Model                  Mask mAP50  IoU micro  Dice micro  MAE kcal  RMSE kcal
-----------------------------------------------------------------------------
YOLO26n-Seg (Ours)          94.08      94.04       96.87     41.60     107.97
YOLOv8n-Seg                 97.15      94.23       96.99     40.29      99.71
FR-CNN+GrabCut                  —      92.50       95.78     65.01     249.85
FR-CNN+SAM                      —      97.50       98.61     63.57     249.81

GAP3 Wilcoxon ITEM-LEVEL (N=41 items, primary — pairs are clustered):
  04_vs_06a: item p=0.4962 | 04 better 24, baseline better 17 (pair-level ref p=6.86e-101)
  04_vs_06b: item p=1.0000 | 04 better 23, baseline better 18 (pair-level ref p=2.32e-88)
  04_vs_07: item p=0.4486 | 04 better 20, baseline better 21 (pair-level ref p=1.43e-02)
